# Tutorial 1 — The data, the split, and the two baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_01_data_and_baselines.ipynb)

Companion to the *Dataset*, *Baseline Popularity*, *ALS* and *Scoring the models* sections of
the post. In ~5 minutes on a free Colab runtime this notebook:

1. loads the Amazon Reviews 2023 **Video_Games 5-core** data the whole project uses,
2. lets you pick a user and read their history,
3. shows why the split is *by time* and what that does to cold-start,
4. lets you play with Recall@K and NDCG@K on toy lists until they make sense,
5. fits the popularity and ALS baselines and lets you compare their recommendations for any user against what that user actually did next.

Everything runs against the project's own code (`backend/models/*.py`), not a re-implementation.

In [ ]:
#@title Setup — clone the repo, install deps, download the pre-computed artifacts (~1 min)
import os, sys, urllib.request
REPO_URL = "https://github.com/juanmigutierrez/generative-recommendation-engine"
RELEASE  = REPO_URL + "/releases/download/v1.0-artifacts"
QUICK    = os.environ.get("TUTORIAL_QUICK") == "1"   # tiny sizes for headless smoke tests

if not os.path.exists("backend"):
    if not os.path.exists("generative-recommendation-engine"):
        !git clone -q {REPO_URL}
    %cd generative-recommendation-engine
if "google.colab" in sys.modules:
    !pip install -q implicit lightgbm pyarrow ipywidgets 2>&1 | tail -1
os.makedirs("data/processed", exist_ok=True)

def fetch(*names):
    """Download an artifact from the GitHub release unless it is already on disk."""
    for n in names:
        p = os.path.join("data", "processed", n)
        if not os.path.exists(p):
            print("downloading", n, "...")
            urllib.request.urlretrieve(f"{RELEASE}/{n}", p)

fetch("train.parquet", "val.parquet", "test.parquet", "item_catalog.parquet", "val_targets.parquet", "test_targets.parquet")
for p in ["backend", "backend/scripts"]:
    if p not in sys.path: sys.path.insert(0, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
pd.set_option("display.max_colwidth", 90)
P = os.path.join("data", "processed")
print("ready")

## 1. The data

One row per (user, item, timestamp). `description` is the product title — the only content signal used in the whole project.

In [ ]:
train = pd.read_parquet(f"{P}/train.parquet")
val   = pd.read_parquet(f"{P}/val.parquet")
test  = pd.read_parquet(f"{P}/test.parquet")
items = pd.read_parquet(f"{P}/item_catalog.parquet").set_index("item_id")
title = items["description"]

n_users = pd.concat([train, val, test])["user_id"].nunique()
print(f"{len(train)+len(val)+len(test):,} interactions, {n_users:,} users, {len(items):,} items")
per_user = pd.concat([train, val, test]).groupby("user_id").size()
print(f"interactions per user: median {per_user.median():.0f}, mean {per_user.mean():.1f}, max {per_user.max()}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(per_user.clip(upper=40), bins=36, color="#2a78d6")
ax.set_xlabel("interactions per user (clipped at 40)"); ax.set_ylabel("users"); ax.set_title("Every user has ≥ 5 by construction (5-core), most have 5–8")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.show()

### Pick a user and read their history

The dropdown holds 30 random users with a longer history so there is something to read. `split` tells you which side of the time cut each review falls on.

In [ ]:
allrows = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
allrows["timestamp"] = pd.to_datetime(allrows["timestamp"])
rng = np.random.RandomState(0)
long_users = per_user[per_user.between(8, 25)].index
sample_users = sorted(rng.choice(long_users, 30, replace=False).tolist())

def show_history(user_id):
    h = allrows[allrows.user_id == user_id].sort_values("timestamp")
    out = pd.DataFrame({"when": h["timestamp"].dt.date.values, "split": h["split"].values,
                        "item_id": h["item_id"].values, "title": title.loc[h["item_id"]].str.slice(0, 80).values})
    display(out.reset_index(drop=True))

interact(show_history, user_id=widgets.Dropdown(options=sample_users, description="user"));

### How one user is scored

![The exam every model takes: history before the cutoff → top-K guesses → what actually happened → Recall@K and NDCG@K.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/ranking_models.png)

*The exam every model takes: history before the cutoff → top-K guesses → what actually happened → Recall@K and NDCG@K.*

## 2. Splitting by time, not at random

A random split lets a model train on a 2022 review and be tested on one from 2005 — it has seen the future. So the cut is by date: the last 8% of the dataset's time span is test, the 8% before that is val, everything earlier is train. The price is that val/test contain users the model has never seen (cold users) and items that did not exist yet (cold items). This is the honest production question — and, as the post explains, a *much* harder one than the papers' leave-one-out protocol (Tutorial 5).

In [ ]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    t = pd.to_datetime(df["timestamp"])
    print(f"{name:<6} {t.min().date()} → {t.max().date()}   {len(df):>8,} rows   {df.user_id.nunique():>7,} users")

train_users, train_items = set(train.user_id), set(train.item_id)
for name, df in [("val", val), ("test", test)]:
    cold_u = (~df.user_id.isin(train_users)).groupby(df.user_id).first().mean()
    cold_i = (~df.item_id.isin(train_items)).mean()
    print(f"{name}: {cold_u:.0%} of users never appear in train; {cold_i:.0%} of rows are items never seen in train")

## 3. The metrics, on toy lists

Both metrics take a ranked list and the set of items the user actually interacted with next.

**Recall@K — did we find it?** Of the items the user actually interacted with, what fraction appear anywhere in the top-K list? Order inside the list doesn't matter.

$$\mathrm{Recall@}K = \frac{|\, \mathrm{top}K \cap \mathrm{relevant} \,|}{|\, \mathrm{relevant} \,|}$$

**NDCG@K — did we find it early?** Each hit gets a weight that shrinks with its position, and the total is divided by the best score possible so the result lands in [0, 1]:

$$\mathrm{DCG@}K = \sum_{i=1}^{K} \frac{\mathbb{1}[\text{item}_i \in \mathrm{relevant}]}{\log_2(i+1)}, \qquad \mathrm{NDCG@}K = \frac{\mathrm{DCG@}K}{\mathrm{IDCG@}K}$$

![Recall@5 on three lists: only presence counts.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/eval_example_recall.png)

*Recall@5 on three lists: only presence counts.*

![NDCG@5: same hit, different position, different score.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/eval_example_ndcg.png)

*NDCG@5: same hit, different position, different score.*

Edit the two lists below and re-run — the functions are the project's own (`backend/models/metrics.py`).

In [ ]:
from models.metrics import recall_at_k, ndcg_at_k

relevant = {"E", "M"}                              #@param {type:"raw"}
ranked   = ["G", "E", "H", "J", "K", "M", "N"]     #@param {type:"raw"}

for k in (3, 5, 7):
    print(f"K={k}: recall@{k} = {recall_at_k(ranked, set(relevant), k):.2f}   ndcg@{k} = {ndcg_at_k(ranked, set(relevant), k):.2f}")
print("\nposition weights used by NDCG (1/log2(pos+1)):", [round(float(1/np.log2(i+2)), 2) for i in range(7)])

In [ ]:
@interact(hit_position=widgets.IntSlider(1, 1, 10, description="hit at #"))
def ndcg_vs_position(hit_position):
    ranked = [f"x{i}" for i in range(10)]; ranked[hit_position - 1] = "E"
    print(f"one relevant item at position {hit_position}:  recall@10 = {recall_at_k(ranked, {'E'}, 10):.2f}   ndcg@10 = {ndcg_at_k(ranked, {'E'}, 10):.2f}")

## 4. Baseline 1 — popularity

Count how often each item appears in train, sort, cross off what the user already has, return the top-K. Everyone gets the same list.

$$\mathrm{pop}(i) = \sum_{u} \mathbb{1}\big[(u, i) \in \mathcal{D}\big], \qquad \mathrm{rec}(u) = \operatorname{top\text{-}K}_{\,i \notin H_u} \; \mathrm{pop}(i)$$

![Count → sort → remove what the user already has → return the top-K. The only personalisation is the crossing-off.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/popularity_model_histogram_indigo.png)

*Count → sort → remove what the user already has → return the top-K. The only personalisation is the crossing-off.*

In [ ]:
from models.popularity import PopularityRecommender
pop = PopularityRecommender().fit(train)
print("the ten most popular items in train:")
for i in pop.ranked_items[:10]:
    print(f"  {title[i][:90]}")

## 5. Baseline 2 — ALS

Put users in rows and items in columns; a 1 where we saw an interaction, a blank where we didn't. Call it $R$. Recommending is filling in the blanks. ALS says $R$ is (approximately) the product of two thin matrices — a row $\mathbf{x}_u$ per user and a row $\mathbf{y}_i$ per item, both of length $k$ — and the prediction for any cell is their dot product:

$$\hat r_{ui} = \mathbf{x}_u^{\top} \mathbf{y}_i$$

![R ≈ X · Yᵀ. The dashed cell (user 1, item 2) was never observed; the model predicts 0.45 for it.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/als_matrices_R_X_Y.png)

*R ≈ X · Yᵀ. The dashed cell (user 1, item 2) was never observed; the model predicts 0.45 for it.*

The numbers are chosen to make the predictions match the cells we know, with a penalty on their size so the model can't overfit by making them huge (regularisation, $\lambda$). And because a blank means *not observed*, not *disliked*, each cell gets a confidence weight — trust the 1s a lot, the blanks only a little (Hu, Koren & Volinsky 2008):

$$c_{ui} = 1 + \alpha\, r_{ui}, \qquad p_{ui} = \mathbb{1}[r_{ui} > 0]$$

$$\min_{X,Y} \sum_{u,i} c_{ui}\,\big(p_{ui} - \mathbf{x}_u^{\top}\mathbf{y}_i\big)^2 + \lambda\Big(\sum_u \|\mathbf{x}_u\|^2 + \sum_i \|\mathbf{y}_i\|^2\Big)$$

Solving for $X$ and $Y$ together is hard, so freeze one and the other has a closed-form least-squares solution; alternate until it stops improving — hence the name:

$$\mathbf{x}_u = (Y^{\top} C^u Y + \lambda I)^{-1} Y^{\top} C^u \mathbf{p}_u, \qquad \mathbf{y}_i = (X^{\top} C^i X + \lambda I)^{-1} X^{\top} C^i \mathbf{p}_i$$

![The three terms of the loss on the toy matrix: confidence, error, regularisation.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/als_full_breakdown_diagram.png)

*The three terms of the loss on the toy matrix: confidence, error, regularisation.*

The `implicit` library implements exactly this. 64 factors, $\alpha = 40$, 15 alternations — about 30 s.

In [ ]:
from models.als import ALSRecommender
import time
n_items = len(items); n_users_total = int(allrows.user_id.max()) + 1
t0 = time.time()
als = ALSRecommender(factors=64, regularization=0.05, iterations=15, alpha=40.0).fit(train, n_users_total, n_items)
print(f"ALS fit in {time.time()-t0:.0f}s")

### What ALS learned: nearest items in its latent space

No content was used — these neighbours come purely from who-bought-what.

In [ ]:
item_vecs = als.model.item_factors
if hasattr(item_vecs, "to_numpy"): item_vecs = item_vecs.to_numpy()   # GPU build returns a wrapper
item_vecs = np.asarray(item_vecs); item_norm = item_vecs / (np.linalg.norm(item_vecs, axis=1, keepdims=True) + 1e-9)
popular_examples = [i for i in pop.ranked_items[:300] if len(title[i]) < 70][:25]

def als_neighbours(item_id):
    sims = item_norm @ item_norm[item_id]
    top = np.argsort(-sims)[1:8]
    print("query:", title[item_id]); print()
    for j in top: print(f"  {sims[j]:.3f}  {title[j][:85]}")

interact(als_neighbours, item_id=widgets.Dropdown(options=[(title[i][:60], i) for i in popular_examples], description="item"));

## 6. Compare the two for one user

For a user with training history, both models produce a top-10; the *val* items are what the user actually reviewed next. Hits are marked. Try several users — you will see how rarely either model hits, which is the point the post makes about this split.

In [ ]:
val_targets = {r.user_id: set(r.item_ids) for r in pd.read_parquet(f"{P}/val_targets.parquet").itertuples()}
warm_val_users = [u for u in val_targets if u in train_users and per_user[u] >= 8]
demo_users = sorted(rng.choice(warm_val_users, 40, replace=False).tolist())

def compare(user_id, K=10):
    truth = val_targets[user_id]
    print("actually reviewed next (val):")
    for i in truth: print("   •", title[i][:85])
    for name, fn in [("popularity", pop.recommend), ("ALS", als.recommend)]:
        recs = fn(user_id, K)
        hits = sum(i in truth for i in recs)
        print(f"\n{name} top-{K}  —  recall@{K} = {hits/len(truth):.2f}, ndcg@{K} = {ndcg_at_k(recs, truth, K):.2f}")
        for r, i in enumerate(recs, 1):
            print(f"  {'✔' if i in truth else ' '} {r:>2}. {title[i][:80]}")

interact(compare, user_id=widgets.Dropdown(options=demo_users, description="user"), K=widgets.IntSlider(10, 5, 20, 5));

## 7. Score both on a sample of val users

The post's baseline table, reproduced on a random 2,000-user sample (cold users fall back to the popularity list for ALS, exactly as in the project's `run_baselines.py`).

In [ ]:
from models.metrics import evaluate
n_eval = 300 if QUICK else 2000
eval_users = rng.choice(sorted(val_targets), n_eval, replace=False)
targets = {u: val_targets[u] for u in eval_users}
als_fn = lambda u, k: als.recommend(u, k) if not als.is_cold(u) else pop.recommend(u, k)
rows = {"popularity": evaluate(pop.recommend, targets), "als": evaluate(als_fn, targets)}
table = pd.DataFrame(rows).T[["recall@10", "recall@20", "ndcg@10", "ndcg@20"]].round(4)
display(table)
print("Low numbers everywhere — and ALS barely beats counting. Tutorial 5 shows the same models under the papers' protocol, where the picture changes completely.")

**Next:** [Tutorial 2 — Semantic IDs](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_02_semantic_ids.ipynb): give every item a code built from its title.